# Paso 4 — Canal Optimization + Final Map
Least-cost drainage paths + retention lagoon siting + crop suitability map.

In [ ]:
import sys
sys.path.insert(0, '..')
from src.config import load_config
from src.canals import least_cost_drainage_paths, identify_retention_sites, crop_suitability_map
from src.viz import final_map, plot_raster
from pathlib import Path

cfg = load_config()
proc = Path('../data/processed')
outputs = Path('../outputs')
outputs.mkdir(exist_ok=True)

## 4.1 Least-cost drainage paths

In [ ]:
drainage = least_cost_drainage_paths(
    flow_acc_path=proc / 'flow_acc.tif',
    threshold_cells=500,
    output_path=str(outputs / 'drainage_paths.gpkg'),
)
print(f'{len(drainage)} drainage channel features')

## 4.2 Retention lagoon sites

In [ ]:
retention = identify_retention_sites(
    sinks_path=proc / 'sinks.tif',
    twi_path=proc / 'twi.tif',
    min_area_pixels=20,
    twi_threshold=10.0,
    output_path=str(outputs / 'retention_sites.gpkg'),
)
print(f'{len(retention)} retention candidate sites')
retention[['area_ha', 'mean_twi']].describe()

## 4.3 Crop suitability map

In [ ]:
crop_suitability_map(
    flood_prob_path=outputs / 'flood_probability.tif',
    output_path=str(outputs / 'crop_suitability.tif'),
)
plot_raster(outputs / 'crop_suitability.tif',
            title='Crop Suitability', cmap='YlGn',
            output_path=str(outputs / 'crop_suitability_map.png'))

## 4.4 Final interactive map

In [ ]:
m = final_map(
    flood_prob_path=outputs / 'flood_probability.tif',
    drainage_gdf=drainage,
    retention_gdf=retention,
    suitability_path=outputs / 'crop_suitability.tif',
    output_html=str(outputs / 'final_map.html'),
)
m